## 391_SAT_ID_ref_stockholmarchipelagotrail
* [#391](https://github.com/salgo60/Stockholm_Archipelago_Trail/issues/391)
* denna Notebook [391_SAT_ID_ref_stockholmarchipelagotrail.ipynb](https://github.com/salgo60/Stockholm_Archipelago_Trail/tree/main/Notebook/391_SAT_ID_ref_stockholmarchipelagotrail.ipynb)

### Statistik  


| Tidpunkt | SAT | OSM `unknown` | OSM med värde |
|----------|----:|--------------:|--------------:|
| 2026-06-13 13:31 | 326 | 126 | 436 |
| 2026-06-13 15:27 | 326 | 137 | 437 |
| 2026-06-13 17:08 | 326 | 139 | 437 |



In [1]:
import time
import datetime  
start_time = time.time()
start_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
print(f"Started: {start_str}")


Started: 2026-06-13 20:06


### antal objekt OSM 
* antal unknow

In [2]:
import requests

query = """
[out:json][timeout:25];
nwr["ref:stockholmarchipelagotrail"="unknown"];
out count;
"""

headers = {
    "User-Agent": "MagnusOSMTools/1.0 (https://github.com/salgo60)"
}

response = requests.post(
    "https://overpass-api.de/api/interpreter",
    data=query,
    headers=headers,
    timeout=60,
)

response.raise_for_status()

data = response.json()
tags = data["elements"][0]["tags"]

print(f"Nodes: {tags['nodes']}")
print(f"Ways: {tags['ways']}")
print(f"Relations: {tags['relations']}")
print(f"Total: {tags['total']}") 


print("-----")

import requests

query = """
[out:json][timeout:25];
nwr["ref:stockholmarchipelagotrail"]["ref:stockholmarchipelagotrail"!="unknown"];
out count;
"""

headers = {
    "User-Agent": "MagnusOSMTools/1.0 (https://github.com/salgo60 salgo60@msn.com)"
}

response = requests.post(
    "https://overpass-api.de/api/interpreter",
    data={"data": query},
    headers=headers,
    timeout=60,
)

response.raise_for_status()

data = response.json()
tags = data["elements"][0]["tags"]

print(f"Nodes: {tags['nodes']}")
print(f"Ways: {tags['ways']}")
print(f"Relations: {tags['relations']}")
print(f"Total: {tags['total']}")

HTTPError: 504 Server Error: Gateway Timeout for url: https://overpass-api.de/api/interpreter

### antal objekt SAT 

In [ ]:
import json
import requests

# Läs GeoJSON
url = "https://map.stockholmarchipelagotrail.com/data/geojson/pois.geojson"
geojson = requests.get(url).json()

errors = []

for feature in geojson["features"]:
    props = feature["properties"]

    sat_id = props["id"]

    # Hitta OSM-referensen
    osm_ref = None
    for item in props.get("sameAs", []):
        if item.startswith("osm:"):
            osm_ref = item
            break

    if osm_ref is None:
        continue

    _, osm_type, osm_id = osm_ref.split(":")

    errors.append({
        "type": osm_type,
        "id": osm_id,
        "expected_ref": sat_id
    })

print(f"Antal OSM-objekt att kontrollera: {len(errors)}")

In [ ]:
from IPython.display import HTML, display

osm_lookup = {e["expected_ref"]: e for e in errors}

rows = []

for feature in geojson["features"]:
    props = feature["properties"]
    sat_id = props["id"]

    if sat_id in osm_lookup:
        status = "✅ ok"
        osm = osm_lookup[sat_id]

        osm_link = (
            f"<a href='https://www.openstreetmap.org/{osm['type']}/{osm['id']}' "
            f"target='_blank'>{osm['type']}/{osm['id']}</a>"
        )

        unknown = (
            "⚠️ unknown"
            if osm.get("osm_ref") == "unknown"
            else ""
        )
    else:
        status = "❌ false"
        osm_link = ""
        unknown = ""

    rows.append(
        f"""
        <tr>
            <td>{status}</td>
            <td>{unknown}</td>
            <td>{sat_id}</td>
            <td>{props.get('name', '')}</td>
            <td>{osm_link}</td>
        </tr>
        """
    )

html = f"""
<table border="1">
    <tr>
        <th>Status</th>
        <th>OSM ref=unknown</th>
        <th>SAT ID</th>
        <th>Namn</th>
        <th>OSM</th>
    </tr>
    {''.join(rows)}
</table>
"""

display(HTML(html))

### Sök ut wd och kolla om koppling osm finns
* kolla om OSM har ref:stockholmarchipelagotrail
* [SPARQL](https://w.wiki/R9Q2)


In [3]:
# saknar OSM 
import requests
import pandas as pd

query = """
SELECT DISTINCT
  ?id
  ?idLabel
  ?geo
WHERE {
  ?id wdt:P6104 wd:Q134294510 ;
      wdt:P625 ?geo .

  FILTER NOT EXISTS { ?id wdt:P11693 ?osmNode . }
  FILTER NOT EXISTS { ?id wdt:P10689 ?osmWay . }
  FILTER NOT EXISTS { ?id wdt:P402 ?osmRelation . }

  OPTIONAL { ?id wdt:P18 ?img . }

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "sv,en,de,en" .
  }
}
"""

url = "https://query.wikidata.org/sparql"
headers = {
    "Accept": "application/sparql-results+json",
    "User-Agent": "MagnusOSMTools/1.0 salgo60@msn.com"
}

response = requests.get(
    url,
    params={"query": query},
    headers=headers
)
response.raise_for_status()

data = response.json()["results"]["bindings"]

rows = []
for row in data:
    rows.append({
        "QID": row["id"]["value"].split("/")[-1],
        "Wikidata": row["id"]["value"],
        "idLabel": row.get("idLabel", {}).get("value", ""),
        "Koordinater": row.get("geo", {}).get("value", ""),
    })

df = pd.DataFrame(rows)

df["Wikidata"] = df["QID"].apply(
    lambda q: f'<a href="https://www.wikidata.org/wiki/{q}" target="_blank">{q}</a>'
)

from IPython.display import HTML
print("Saknar OSM objekt")
HTML(df.to_html(escape=False, index=False))

Saknar OSM objekt


QID,Wikidata,idLabel,Koordinater
Q135217025,Q135217025,Villa Söderängen,Point(18.275727 58.9506268)
Q135107443,Q135107443,Svartsö Sjöbodar,Point(18.656293 59.438004)
Q115371401,Q115371401,Arholma kyrkogård,Point(19.110433 59.846011)
Q32301589,Q32301589,Nåttaröhals,Point(18.104166666 58.858333333)
Q136756996,Q136756996,Gamla skolhuset,Point(18.321663 58.969196)
Q134580717,Q134580717,Solberga gård Runmarö kajak uthyrning,Point(18.758125 59.255867)
Q1480015,Q1480015,Landsort,Point(17.8658 58.739603)
Q134839262,Q134839262,"Ornö badplats, Kyrkviken",Point(18.436481019 59.055193597)
Q135443360,Q135443360,Ramsmora badklippor,Point(18.917464413 59.429324795)
Q31895480,Q31895480,Edesnäs,Point(18.280694 58.954111)


In [ ]:
import requests
import time
from tqdm.notebook import tqdm

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

headers = {
    "User-Agent": "MagnusOSMTools/1.0 (salgo60@msn.com)"
}

osm_ref = []
osm_link = []

for qid in tqdm(df["QID"], total=len(df), desc="Kontrollerar OSM"):
    query = f"""
    [out:json][timeout:50];
    nwr["ref:stockholmarchipelagotrail"="{qid}"];
    out ids;
    """

    try:
        r = requests.post(OVERPASS_URL, data=query, headers=headers)
        r.raise_for_status()
        elements = r.json().get("elements", [])

        if elements:
            e = elements[0]
            osm_type = e["type"]  # node, way eller relation
            osm_id = e["id"]

            osm_ref.append("Ja")
            osm_link.append(
                f'<a href="https://www.openstreetmap.org/{osm_type}/{osm_id}" '
                f'target="_blank">{osm_type}/{osm_id}</a>'
            )
        else:
            osm_ref.append("Nej")
            osm_link.append("")

    except Exception as ex:
        osm_ref.append(f"Fel: {ex}")
        osm_link.append("")

    # Var snäll mot Overpass-servern
    time.sleep(4)

df["OSM_ref"] = osm_ref
df["OSM_länk"] = osm_link

from IPython.display import HTML
HTML(df.to_html(escape=False, index=False))

In [ ]:
end_time = time.time()
duration = end_time - start_time

print(f"Fini§shed in {duration:.2f} seconds.")
